## Opening Dataset from preprocessing and loading libraries

In [21]:
from bertopic import BERTopic
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np  
import umap
import hdbscan
from sentence_transformers import SentenceTransformer
from google import genai
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI, PartOfSpeech
import spacy 
from umap import UMAP
from hdbscan import HDBSCAN
# Load the data

df = pd.read_csv('C:\\Users\\20193623\\OneDrive - TU Eindhoven\\BEP\\Mijn project\\data\\cleaned\\df_long_clean.csv')

#Dit werkte dus niet 
# docs = df_filtered['merged_lies'].tolist()
# docs_positive = df_positive['merged_lies'].tolist()
# docs_negative = df_negative['merged_lies'].tolist()

BELANGRIJK STAP VOOR HET OMZETTEN NAAR DOCS -> De merged lies column werkte niet want je wil juist alle losse leugens als input en niet de som van totale input want dan worden de verschillende topics in de leugens als samenhangend gezien en dat is niet


In [22]:
# Create a list of your unique events
unique_events = df['Event'].unique().tolist()
print(unique_events)

['A job interview for your dream job', 'Getting fired', 'Being hospitalised and undergoing surgery', 'Missing a deadline at work because of bad organisation', 'Taking the bus/train without the ticket', 'Getting a speeding fine', 'Being involved in a car accident', 'Causing a car accident', 'Cheating on an exam']


In [23]:

# # Define keywords for each event. This is a crucial manual step.
# # Replace these example keywords with words relevant to your specific events.
# # The number of inner lists must match the number of unique events.
# seed_topic_list = [
#     ['job', 'work', 'career', 'interview', 'fired', 'terminated', 'deadline', 'project', 'task'], # Carrièregebeurtenissen
#     ['car', 'accident', 'crash', 'collision', 'speeding', 'fine', 'traffic'],                   # Auto & Verkeer
#     ['hospital', 'surgery', 'operation', 'health', 'illness', 'doctor'],                        # Zorg & Medische Gebeurtenissen
#     ['cheating', 'exam', 'ticket', 'fine', 'illegal', 'rule', 'punishment']                     # Overtredingen & Gevolgen
# ]

In [24]:
docs = df['enriched_lie'].tolist()

# Start embedding and saving them for optimization


In [25]:
embedding_model = SentenceTransformer("all-mpnet-base-v2")
embeddings = embedding_model.encode(docs, show_progress_bar=True)

Batches: 100%|██████████| 94/94 [02:13<00:00,  1.42s/it]


In [26]:
# UMAP
umap_n_neighbors = 25
umap_n_components = 10   # meer dimensies -> subtielere structuur
umap_min_dist = 0.3
umap_metric = "cosine"

# HDBSCAN
hdbscan_min_cluster_size = 20
hdbscan_metric = "euclidean"
hdbscan_cluster_selection_method = "eom"

# CountVectorizer
vectorizer_min_df = 2
vectorizer_ngram_range = (1, 3)  # langere zinsdelen meenemen


umap_model = UMAP(
    n_neighbors=umap_n_neighbors,
    n_components=umap_n_components,
    min_dist=umap_min_dist,
    metric=umap_metric,
    random_state=umap_random_state
)

hdbscan_model = HDBSCAN(
    min_cluster_size=hdbscan_min_cluster_size,
    metric=hdbscan_metric,
    cluster_selection_method=hdbscan_cluster_selection_method,
    prediction_data=hdbscan_prediction_data
)

vectorizer_model = CountVectorizer(
    stop_words=vectorizer_stop_words,
    min_df=vectorizer_min_df,
    ngram_range=vectorizer_ngram_range
)

In [27]:

# KeyBERT
keybert_model = KeyBERTInspired()

# Part-of-Speech
pos_model = PartOfSpeech("en_core_web_sm")

# MMR
mmr_model = MaximalMarginalRelevance(diversity=0.3)

# All representation models
representation_model = {
    "KeyBERT": keybert_model,
    "MMR": mmr_model,
    "POS": pos_model
}

# Initializing BERTopic from pipeline

In [28]:
from bertopic import BERTopic

topic_model = BERTopic(

  # Pipeline models
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  representation_model=representation_model,

  # Hyperparameters
  top_n_words=10, #visual number of words per topic
  verbose=True,

  # seed_topic_list=seed_topic_list
)

# Train model
topics, probs = topic_model.fit_transform(docs, embeddings)

# Show topics
topic_model.get_topic_info()
topic_model.visualize_documents(docs)

2025-10-09 10:13:41,981 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-10-09 10:14:07,316 - BERTopic - Dimensionality - Completed ✓
2025-10-09 10:14:07,326 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-10-09 10:14:07,865 - BERTopic - Cluster - Completed ✓
2025-10-09 10:14:07,893 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-10-09 10:14:37,024 - BERTopic - Representation - Completed ✓


In [29]:
hierarchical_topics = topic_model.hierarchical_topics(docs)

topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)


100%|██████████| 11/11 [00:00<00:00, 166.15it/s]


In [30]:
topic_model.get_topic_info()


,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,1647,-1_interview_work_day_did,"[interview, work, day, did, got, time, job, to...","[job interview, interview, loss, gave, didn, h...","[interview, work, job, company, ago, friend, h...","[interview, day, time, job, company, months, f...",[I applied for a job and was called to an inte...
1,0,531,0_car_ticket_driving_bus,"[car, ticket, driving, bus, train, driver, roa...","[car accident, drove, accident, driving, car, ...","[car, driving, bus, road, traffic, speeding, t...","[car, ticket, driving, bus, train, driver, roa...","[I was driving the car., A morning i was drivi..."
2,1,258,1_interview_job_dream_dream job,"[interview, job, dream, dream job, job intervi...","[interview dream job, interview dream, job int...","[interview, job, dream job, job interview, int...","[interview, job, dream, position, life, excite...",[The job interview for my dream job came at su...
3,2,194,2_surgery_pain_doctor_hospital,"[surgery, pain, doctor, hospital, hours, woke,...","[undergo tonsilectomy, tonsilectomy sudden, to...","[surgery, hospital, hours, doctors, cyst, infe...","[surgery, pain, doctor, hospital, hours, docto...",[IN SEPTEMBER 2023 I WAS HOSPITALISED AND HAD ...
4,3,105,3_exam_questions_notes_second,"[exam, questions, notes, second, studying, stu...","[exam studied, exam, prepared exam, studying, ...","[exam, notes, studying, studied, writing, reme...","[exam, questions, notes, second, bit, test, we...",[It was my first exam of this semester. I was ...
5,4,76,4_deadline_work_task_report,"[deadline, work, task, report, instructions, t...","[failed meet deadline, work unexpected circums...","[report, meet deadline, complete task, deadlin...","[deadline, work, task, report, instructions, t...",[I failed to meet a deadline at work due to un...
6,5,41,5_told_listen_just said_uncle,"[told, listen, just said, uncle, mother, said,...","[told granddad, granddad swapped train, told g...","[listen, looked, room, informed decision, golf...","[uncle, mother, question, situation, room, gol...",[told me that he hopes that it's the last time...
7,6,31,6_boss_want_manager_told,"[boss, want, manager, told, team, said, superv...","[boss got upset, got upset reprimanded, boss, ...","[boss, want, manager, told, supervisor, team a...","[boss, manager, team, supervisor, employee, bi...","[the boss told me I suck at my job, They both ..."
8,7,28,7_said_explain_talking_trying,"[said, explain, talking, trying, didn, doing, ...","[woman, didn greet, mood didn greet, lady, was...","[talking, husband, mood didn greet, catch atte...","[care, husband, memory, good mood, words, bad,...",[d I went home excited. when I got home I foun...
9,8,26,8_panicking_bad_excited_remember,"[panicking, bad, excited, remember, chose lot ...","[took away thoughts, remember crying toilets, ...","[panicking, chose lot thought, actually excite...","[bad, excited, anger, stupid, good idea, bad t...","[I chose not to give that a lot of thought, I ..."
